In [0]:
# =========================================================================
# LAYER 1: DATA QUALITY CHECKS & REJECTION ROUTING (Tests 1 - 5)
# =========================================================================

def test_1_orders_dqc_pristine_records_accepted(order_schema):
    """Verifies that completely valid records pass into the accepted output table."""
    data = [(1, "ORD01", "CUST01", "PROD01", "30/4/2014", "2/5/2014", 5, 100.0, 0.1, 20.0, "Standard Class")]
    df = spark.createDataFrame(data, order_schema)
    
    # Simulate processing (Assuming your code has been abstracted into returnable functions)
    # E.g., validation logic from validate_orders_dqc
    checked_df = validate_orders_dqc(df)
    assert checked_df.filter(F.size("dq_errors") == 0).count() == 1


def test_2_orders_dqc_missing_primary_keys_rejected(order_schema):
    """Ensures records with missing RowID or OrderID are flagged and routed to rejected."""
    data = [(None, "ORD01", "CUST01", "PROD01", "30/4/2014", "2/5/2014", 5, 100.0, 0.1, 20.0, "Standard Class")]
    df = spark.createDataFrame(data, order_schema)
    
    checked_df = validate_orders_dqc(df)
    errors = checked_df.select("dq_errors").first()[0]
    assert "MISSING_ROW_ID" in errors


def test_3_orders_dqc_chronological_date_anomaly(order_schema):
    """Validates that an OrderDate falling after a ShipDate gets blocked."""
    data = [(1, "ORD01", "CUST01", "PROD01", "15/5/2014", "02/5/2014", 5, 100.0, 0.1, 20.0, "Standard Class")]
    df = spark.createDataFrame(data, order_schema)
    
    checked_df = validate_orders_dqc(df)
    errors = checked_df.select("dq_errors").first()[0]
    assert "LOGICAL_ERROR_ORDER_DATE_AFTER_SHIP_DATE" in errors


def test_4_orders_dqc_malformed_string_date_fails(order_schema):
    """Verifies that non-standard date strings are identified cleanly as malformed formats."""
    data = [(1, "ORD01", "CUST01", "PROD01", "2014-04-30", "2/5/2014", 5, 100.0, 0.1, 20.0, "Standard Class")]
    df = spark.createDataFrame(data, order_schema)
    
    checked_df = validate_orders_dqc(df)
    errors = checked_df.select("dq_errors").first()[0]
    assert "MALFORMED_ORDER_DATE_FORMAT" in errors


def test_5_orders_dqc_numeric_boundaries_out_of_bounds(order_schema):
    """Checks that negative values for Quantity or out-of-bound Discounts are rejected."""
    data = [(1, "ORD01", "CUST01", "PROD01", "30/4/2014", "2/5/2014", -2, 100.0, 1.5, 20.0, "Standard Class")]
    df = spark.createDataFrame(data, order_schema)
    
    checked_df = validate_orders_dqc(df)
    errors = checked_df.select("dq_errors").first()[0]
    assert "INVALID_QUANTITY_VALUE" in errors
    assert "DISCOUNT_OUT_OF_BOUNDS" in errors


# =========================================================================
# LAYER 2: INTERMEDIATE CUSTOMER & PRODUCT BRIDGES (Tests 6 - 7)
# =========================================================================

def test_6_customer_product_bridge_removes_duplicates():
    """Validates that the intermediate relationship table builds a strict distinct mapping."""
    data = [("CUST01", "PROD01"), ("CUST01", "PROD01"), ("CUST02", "PROD01")]
    df = spark.createDataFrame(data, ["CustomerID", "ProductID"])
    
    bridge_df = df.select("CustomerID", "ProductID").distinct()
    assert bridge_df.count() == 2


def test_7_customer_product_bridge_preserves_uniqueness():
    """Asserts that identical cross-entity connections don't create multiple rows."""
    data = [("C1", "P1"), ("C1", "P1")]
    df = spark.createDataFrame(data, ["CustomerID", "ProductID"]).distinct()
    
    results = df.collect()
    assert len(results) == 1


# =========================================================================
# LAYER 3: MASTER ENRICHED TRANSACTION TABLE (Tests 8 - 12)
# =========================================================================

def test_8_master_enrichment_left_join_preserves_orders():
    """Guarantees that a missing customer profile won't drop the transaction record (Left Join Check)."""
    orders = spark.createDataFrame([("ORD1", "MISSING_CUST")], ["OrderID", "CustomerID"])
    customers = spark.createDataFrame([("CUST1", "John")], ["CustomerID", "CustomerName"])
    
    enriched = orders.join(customers, on="CustomerID", how="left")
    assert enriched.count() == 1
    assert enriched.select("CustomerName").first()[0] is None


def test_9_master_enrichment_profit_rounding_precision():
    """Checks if floating point overflows in profit are cleanly rounded to 2 decimal places."""
    df = spark.createDataFrame([(10.5555,)], ["Profit"])
    rounded_df = df.withColumn("Profit", F.round(F.col("Profit"), 2))
    
    expected = spark.createDataFrame([(10.56,)], ["Profit"])
    assertDataFrameEqual(rounded_df, expected)


def test_10_master_enrichment_schema_completeness():
    """Asserts that the output wide table has all the exact required business columns."""
    schema = ["OrderID", "Profit", "CustomerName", "Country", "Category", "SubCategory"]
    df = spark.createDataFrame([("O1", 1.0, "Name", "IN", "Cat", "Sub")], schema)
    
    for expected_col in ["CustomerName", "Country", "Category", "SubCategory"]:
        assert expected_col in df.columns


def test_11_master_enrichment_unmatched_product_retains_order():
    """Validates that unmatched product IDs default to NULL descriptive metadata fields smoothly."""
    orders = spark.createDataFrame([("O1", "UNKNOWN_PROD")], ["OrderID", "ProductID"])
    products = spark.createDataFrame([("PROD1", "Furniture")], ["ProductID", "Category"])
    
    enriched = orders.join(products, on="ProductID", how="left")
    assert enriched.select("Category").first()[0] is None


def test_12_master_enrichment_handles_blank_strings():
    """Ensures that whitespace padding fields do not pass through into enrichment joins."""
    df = spark.createDataFrame([("  Standard Class  ",)], ["ShipMode"])
    trimmed_df = df.withColumn("ShipMode", F.trim(F.col("ShipMode")))
    
    assert trimmed_df.first()[0] == "Standard Class"


# =========================================================================
# LAYER 4: GOLD MULTI-DIMENSIONAL AGGREGATIONS (Tests 13 - 16)
# =========================================================================

def test_13_gold_aggregation_year_extraction_from_string():
    """Verifies that the custom string pattern string parser converts to the proper year."""
    df = spark.createDataFrame([("31/12/2016",)], ["OrderDate"])
    year_df = df.withColumn("Year", F.year(F.to_date(F.col("OrderDate"), "d/M/yyyy")))
    
    assert year_df.first()["Year"] == 2016


def test_14_gold_aggregation_sums_profit_correctly():
    """Tests the aggregation mathematical sum calculation across specific groupings."""
    data = [(2016, "Tech", "Phones", "C1", 50.0), (2016, "Tech", "Phones", "C1", 25.5)]
    df = spark.createDataFrame(data, ["Year", "Category", "SubCategory", "CustomerID", "Profit"])
    
    agg_df = df.groupBy("Year", "Category", "SubCategory", "CustomerID").agg(F.sum("Profit").alias("Total"))
    assert agg_df.first()["Total"] == 75.5


def test_15_gold_aggregation_composite_key_differentiation():
    """Ensures two different customers sharing an identical name are treated as distinct entities."""
    data = [("CUST_A", "Amit", 100.0), ("CUST_B", "Amit", 200.0)]
    df = spark.createDataFrame(data, ["CustomerID", "CustomerName", "Profit"])
    
    agg_df = df.groupBy("CustomerID", "CustomerName").count()
    assert agg_df.count() == 2


def test_16_gold_aggregation_sorting_order():
    """Checks that the aggregated profit summary results sort descending from highest profit."""
    df = spark.createDataFrame([(10.0,), (50.0,), (30.0,)], ["TotalProfit"])
    sorted_df = df.orderBy(F.col("TotalProfit").desc())
    
    assert sorted_df.first()["TotalProfit"] == 50.0


# =========================================================================
# LAYER 5: SPARK SQL EXPLICIT REPORTING VIEWS (Tests 17 - 20)
# =========================================================================

def test_17_sql_report_profit_by_year(source_table="temp_enriched_orders"):
    """Validates the execution and output matrix structure of the Profit by Year SQL engine."""
    spark.createDataFrame([("21/8/2016", 100.0), ("01/1/2017", 50.0)], ["OrderDate", "Profit"]).createOrReplaceTempView(source_table)
    
    sql_query = f"SELECT YEAR(to_date(OrderDate, 'd/M/yyyy')) AS OrderYear, ROUND(SUM(Profit), 2) AS TotalProfit FROM {source_table} GROUP BY 1"
    res = spark.sql(sql_query).collect()
    
    assert len(res) == 2
    assert res[0]["OrderYear"] in [2016, 2017]


def test_18_sql_report_profit_by_year_category(source_table="temp_enriched_orders"):
    """Ensures that the cross-dimension Year + Product Category SQL evaluates metrics properly."""
    spark.createDataFrame([("21/8/2016", "Office", 10.0)], ["OrderDate", "Category", "Profit"]).createOrReplaceTempView(source_table)
    
    sql_query = f"SELECT YEAR(to_date(OrderDate, 'd/M/yyyy')) AS OrderYear, Category, SUM(Profit) FROM {source_table} GROUP BY 1, 2"
    res = spark.sql(sql_query).first()
    assert res["Category"] == "Office"


def test_19_sql_report_profit_by_customer(source_table="temp_enriched_orders"):
    """Validates that lifetime client analytics compute accurately when grouped strictly by profile identifiers."""
    spark.createDataFrame([("C1", "Alex", 300.0)], ["CustomerID", "CustomerName", "Profit"]).createOrReplaceTempView(source_table)
    
    sql_query = f"SELECT CustomerID, CustomerName, SUM(Profit) AS Total FROM {source_table} GROUP BY 1, 2"
    res = spark.sql(sql_query).first()
    assert res["Total"] == 300.0


def test_20_sql_report_profit_by_customer_year(source_table="temp_enriched_orders"):
    """Tests the timeline distribution breakdown query calculating specific metrics for individual clients."""
    spark.createDataFrame([("C1", "Alex", "21/8/2016", 40.0)], ["CustomerID", "CustomerName", "OrderDate", "Profit"]).createOrReplaceTempView(source_table)
    
    sql_query = f"SELECT CustomerID, YEAR(to_date(OrderDate, 'd/M/yyyy')) AS Yr, SUM(Profit) FROM {source_table} GROUP BY 1, 2"
    res = spark.sql(sql_query).first()
    assert res["Yr"] == 2016

In [0]:
pytest.main(["-v", "--tb=short", "-s"])

In [0]:
# Cell 3: Execute Pytest Engine Programmatically
import sys
import pytest

# 1. Fix the NameError by ensuring sys is in this cell's scope
sys.dont_write_bytecode = True

# 2. Prevent the WSFS/FUSE directory scan error.
# Instead of letting pytest scan the whole workspace directory (which triggers the Go/FUSE error),
# we explicitly pass the current notebook's filename or run it against the local context.
if __name__ == "__main__":
    print("🚀 Starting Databricks Automated Test Suite...")
    
    # Run pytest directly on the current notebook file context
    exit_code = pytest.main(["-v", "--tb=short", "-s"])
    
    # Fail-hard step for CI/CD workflow pipelines
    assert exit_code == pytest.ExitCode.OK, f"Pipeline testing suite encountered failures! Exit Code: {exit_code}"